# Fake News Detection Agent

A LangGraph-based agentic pipeline for detecting fake news. This notebook implements a **two-phase classification workflow** combining traditional Machine Learning (Phase 1) with an autonomous **LLM ReACT Agent** (Phase 2). The pipeline evaluates both stylistic features and contextual truth, finalizing its decision through DeepEval reasoning verification and weighted aggregation.

**Pipeline stages covered in this notebook:**
1. Setup & data loading
2. Data preprocessing (V2 canonical deduplication)
3. Training of 4 candidate ML models
4. Evaluation with metrics and visualisations
5. Model selection with written justification
6. Inference on a new input
7. Downstream skill output (LLM fact-checking + DeepEval + aggregation)
8. LangGraph pipeline definition with all SKILL.md content
9. Gradio interface launch with `share=True`

---
## 1. Setup — Clone Repo & Install Dependencies

In [1]:
# Clone the repository
!git clone https://github.com/SmridhVarma/Fake-News-Detection-Agent.git
%cd Fake-News-Detection-Agent

Cloning into 'Fake-News-Detection-Agent'...
remote: Enumerating objects: 365, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (26/26), done.
remote: Total 365 (delta 12), reused 25 (delta 10), pack-reused 329 (from 1)
Receiving objects: 100% (365/365), 42.78 MiB | 12.17 MiB/s, done.
Resolving deltas: 100% (179/179), done.
Updating files: 100% (88/88), done.
/content/Fake-News-Detection-Agent


In [2]:
# Install all dependencies
!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.7/88.7 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 844.6/844.6 kB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 226.9/226.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.7/40.7 kB 1.9 MB/s eta 0:00:00


In [3]:
# Download NLTK data required by TextBlob and preprocessing
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)
print("NLTK data downloaded.")

NLTK data downloaded.


---
## 2. API Keys (No Hardcoded Keys)

Set your OpenAI and NewsAPI keys. Two options:
- **Option A (Recommended):** Use Colab Secrets — add `OPENAI_API_KEY` and `NEWS_API_KEY` via the key icon in the left sidebar.
- **Option B:** Paste them directly in the cell below (uncomment the lines).

In [4]:
import os

# Option A: Colab Secrets (recommended — no keys in code)
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["NEWS_API_KEY"] = userdata.get("NEWS_API_KEY")
    print("Loaded API keys from Colab Secrets.")
except Exception:
    print("Colab Secrets not available. Use Option B below.")

# Option B: Paste directly (uncomment and fill in)
# os.environ["OPENAI_API_KEY"] = "sk-..."
# os.environ["NEWS_API_KEY"] = "..."

# Verify keys are set
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY not set! Please set OPENAI key."
assert os.environ.get("NEWS_API_KEY"), "NEWS_API_KEY not set! Please set NEWSAPI key."
print("API keys verified.")

Loaded API keys from Colab Secrets.
API keys verified.


---
## 3. Data Loading

The pipeline uses the [Kaggle Fake and Real News Dataset](https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset) containing 44,898 labelled articles (2015–2018).

**Option A:** Download via Kaggle API  
**Option B:** Upload manually

In [5]:
# Option A: Kaggle API download
# If using Kaggle API, first upload your kaggle.json:
#   from google.colab import files
#   files.upload()  # upload kaggle.json
#   !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json

!pip install -q kaggle
!mkdir -p data
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset -p data/ --unzip

!ls -lh data/

Dataset URL: https://www.kaggle.com/datasets/clmentbisaillon/fake-and-real-news-dataset
License(s): CC-BY-NC-SA-4.0
100% 41.0M/41.0M [00:00<00:00, 142MB/s] 

total 111M
-rw-r--r-- 1 root root 60M Apr 18 01:31 Fake.csv
-rw-r--r-- 1 root root 52M Apr 18 01:31 True.csv


In [6]:
# Option B: Manual upload (uncomment if Kaggle API doesn't work)
# from google.colab import files
# !mkdir -p data
# uploaded = files.upload()  # upload Fake.csv and True.csv
# for filename in uploaded:
#     !mv "{filename}" data/
# !ls -lh data/

In [7]:
# Inspect the raw data
import pandas as pd

df_fake = pd.read_csv("./data/Fake.csv")
df_true = pd.read_csv("./data/True.csv")

print(f"Fake articles: {len(df_fake):,}")
print(f"Real articles: {len(df_true):,}")
print(f"Total:         {len(df_fake) + len(df_true):,}")
print(f"\nFake columns: {list(df_fake.columns)}")
print(f"Real columns: {list(df_true.columns)}")
print("\n--- Sample Fake Article ---")
print(df_fake["text"].iloc[0][:300])
print("\n--- Sample Real Article ---")
print(df_true["text"].iloc[0][:300])

Fake articles: 23,481
Real articles: 21,417
Total:         44,898

Fake columns: ['title', 'text', 'subject', 'date']
Real columns: ['title', 'text', 'subject', 'date']

--- Sample Fake Article ---
Donald Trump just couldn t wish all Americans a Happy New Year and leave it at that. Instead, he had to give a shout out to his enemies, haters and  the very dishonest fake news media.  The former reality show star had just one job to do and he couldn t do it. As our Country rapidly grows stronger a

--- Sample Real Article ---
WASHINGTON (Reuters) - The head of a conservative Republican faction in the U.S. Congress, who voted this month for a huge expansion of the national debt to pay for tax cuts, called himself a “fiscal conservative” on Sunday and urged budget restraint in 2018. In keeping with a sharp pivot under way 


---
## 4. Data Preprocessing (V2 Canonical Deduplication)

V2 preprocessing applies **stronger canonical deduplication** — normalising unicode, stripping punctuation, and lowercasing before dedup to catch rows that differ only by formatting. This removes ~6,061 additional duplicates compared to V1's naive `drop_duplicates`.

The preprocessing pipeline:
1. Loads and merges Fake.csv (label=0) and True.csv (label=1)
2. Concatenates title + text into `raw_text`
3. Removes empty rows and canonical duplicates
4. Produces dual-cleaned text: `text_ml` (aggressive normalisation for TF-IDF) and `text_llm` (light cleaning for LLM)
5. Engineers 5 handcrafted stylistic features: `sub_variance`, `mean_subjectivity`, `lexical_density`, `caps_ratio`, `has_dateline`
6. Stratified 3-way split: 70% train / 20% validation / 10% test

In [27]:
import os

PREPROCESSING_ARTIFACT_PATH = "./models/v2/preprocessing_artifacts.joblib"

if os.path.exists(PREPROCESSING_ARTIFACT_PATH):
    print(f"Cached preprocessing artifacts found at {PREPROCESSING_ARTIFACT_PATH} — skipping.")
    preprocess_output = {"preprocessing_artifact_path": PREPROCESSING_ARTIFACT_PATH}
else:
    from src.ml.preprocess_data_v2 import preprocess_data_node

    preprocess_output = preprocess_data_node({
        "fake_csv_path": "./data/Fake.csv",
        "true_csv_path": "./data/True.csv",
        "train_size": 0.70,
        "val_size": 0.20,
        "test_size": 0.10,
        "random_state": 42,
    })

    print("\n=== Preprocessing Results ===")
    print(f"Total rows after dedup:   {preprocess_output['preprocessed_rows']:,}")
    print(f"Duplicates removed:       {preprocess_output['duplicate_rows_removed']:,}")
    print(f"Empty rows removed:       {preprocess_output['empty_rows_removed']}")
    print(f"Train split:              {preprocess_output['train_rows']:,}")
    print(f"Validation split:         {preprocess_output['val_rows']:,}")
    print(f"Test split:               {preprocess_output['test_rows']:,}")
    print(f"Numeric features:         {preprocess_output['numeric_feature_cols']}")
    print(f"Artifact saved to:        {preprocess_output['preprocessing_artifact_path']}")

Cached preprocessing artifacts found at ./models/v2/preprocessing_artifacts.joblib — skipping.


---
## 5. Training of Candidate Models

We train **4 candidate ML models**, all sharing the same feature pipeline:
- **TF-IDF** vectorizer: `max_features=20000`, `ngram_range=(1,2)`, `min_df=3`, `max_df=0.95`
- **StandardScaler** on 5 handcrafted numeric features
- Final feature matrix: horizontal stack of TF-IDF sparse matrix + scaled numeric features

### Candidate Models
| Model | Key Configuration |
|-------|------------------|
| **Logistic Regression** | GridSearchCV over C={0.5, 1.0, 2.0}, solver={liblinear, lbfgs} |
| **SVM (LinearSVC)** | GridSearchCV over C={0.5, 1.0, 2.0}, wrapped in CalibratedClassifierCV for probability output |
| **Random Forest** | GridSearchCV over n_estimators={200, 300}, max_depth={None, 20}, min_samples_leaf={1, 2} |
| **Neural Network (MLP)** | GridSearchCV over hidden_layer_sizes={(128,), (64,128)}, alpha={1e-4, 1e-3} |

All tuning uses 3-fold cross-validation scored on F1.

In [9]:
import os

TRAINING_ARTIFACT_PATH = "./models/v2/training_artifacts.joblib"

if os.path.exists(TRAINING_ARTIFACT_PATH):
    print(f"Cached training artifacts found at {TRAINING_ARTIFACT_PATH} — skipping.")
    from src.utils.training_artifacts import load_artifacts
    cached = load_artifacts(TRAINING_ARTIFACT_PATH)
    train_output = {
        "selected_model_name": cached["selected_model_name"],
        "tuning_enabled": cached.get("tuning_enabled", True),
        "best_params_by_model": cached.get("best_params_by_model", {}),
        "cv_best_scores": cached.get("cv_best_scores", {}),
    }
    print(f"Selected model: {train_output['selected_model_name']}")
else:
    from src.ml.training2 import training_node

    train_output = training_node({
        "preprocessing_artifact_path": preprocess_output["preprocessing_artifact_path"],
        "training_artifact_path": TRAINING_ARTIFACT_PATH,
        "model_dir": "./models/v2",
        "enable_tuning": True,
        "cv_folds": 3,
        "grid_n_jobs": -1,
    })

    print("\n=== Training Results ===")
    print(f"Selected model: {train_output['selected_model_name']}")
    print(f"Tuning enabled: {train_output['tuning_enabled']}")
    print(f"\nBest hyperparameters per model:")
    for name, params in train_output.get('best_params_by_model', {}).items():
        cv_score = train_output.get('cv_best_scores', {}).get(name, 'N/A')
        print(f"  {name}: {params} (CV F1: {cv_score})")

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Fitting 3 folds for each of 3 candidates, totalling 9 fits
Fitting 3 folds for each of 8 candidates, totalling 24 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(


Fitting 3 folds for each of 4 candidates, totalling 12 fits


/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(



=== Training Results ===
Selected model: neural_network
Tuning ena`bled: True

Best hyperparameters per model:
  logistic_regression: {'C': 2.0, 'solver': 'liblinear'} (CV F1: 0.9854)
  svm: {'C': 1.0} (CV F1: 0.9905)
  random_forest: {'max_depth': None, 'min_samples_leaf': 1, 'n_estimators': 300} (CV F1: 0.9797)
  neural_network: {'alpha': 0.0001, 'hidden_layer_sizes': (64, 128)} (CV F1: 0.9913)


---
## 6. Evaluation with Metrics and Visualisations

We evaluate all 4 candidates on both the **validation** and **test** splits using:
- Accuracy, Precision, Recall, F1-score, AUC-ROC
- Confusion matrices for each model
- ROC curve comparison overlay
- Metric comparison bar charts

The evaluation script also computes a **majority baseline** for comparison.

In [10]:
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from collections import Counter
from scipy.sparse import hstack, csr_matrix
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve,
)
from src.utils.training_artifacts import load_artifacts, load_model

# Load training artifacts
TRAINING_ARTIFACT_PATH = "./models/v2/training_artifacts.joblib"
artifacts = load_artifacts(TRAINING_ARTIFACT_PATH)

train_df = artifacts["train_df"]
val_df = artifacts["val_df"]
test_df = artifacts["test_df"]
numeric_feature_cols = artifacts["numeric_feature_cols"]
tfidf = artifacts["tfidf_vectorizer"]
scaler = artifacts["numeric_scaler"]
saved_model_paths = artifacts["saved_model_paths"]
candidate_validation_results = artifacts["candidate_validation_results"]
candidate_test_results = artifacts["candidate_test_results"]

print(f"Training set:   {len(train_df):,} rows")
print(f"Validation set: {len(val_df):,} rows")
print(f"Test set:       {len(test_df):,} rows")
print(f"Models saved:   {list(saved_model_paths.keys())}")

Training set:   27,179 rows
Validation set: 7,765 rows
Test set:       3,884 rows
Models saved:   ['logistic_regression', 'svm', 'random_forest', 'neural_network']


In [11]:
# Rebuild test feature matrices
X_text_test = test_df["text_ml"]
X_num_test = test_df[numeric_feature_cols].copy()
y_test = test_df["label"].astype(int)

X_test_tfidf = tfidf.transform(X_text_test)
X_num_test_scaled = scaler.transform(X_num_test)
X_test_full = hstack([X_test_tfidf, csr_matrix(X_num_test_scaled)])

# Majority baseline
majority_class = Counter(train_df["label"]).most_common(1)[0][0]
majority_accuracy = (y_test == majority_class).mean()
print(f"Majority baseline accuracy: {majority_accuracy:.4f}")

Majority baseline accuracy: 0.5391


In [12]:
# Display validation metrics as a table
print("=== Validation Metrics (used for model selection) ===")
val_df_display = pd.DataFrame([
    {"Model": name.replace('_', ' ').title(), **metrics}
    for name, metrics in candidate_validation_results.items()
]).sort_values("f1", ascending=False)
display(val_df_display)

print("\n=== Test Metrics (held-out generalization estimate) ===")
test_df_display = pd.DataFrame([
    {"Model": name.replace('_', ' ').title(), **metrics}
    for name, metrics in candidate_test_results.items()
]).sort_values("f1", ascending=False)
display(test_df_display)

=== Validation Metrics (used for model selection) ===


,Model,accuracy,precision,recall,f1,auc_roc
3,Neural Network,0.9920,0.9907,0.9945,0.9926,0.9993
1,Svm,0.9916,0.9905,0.9940,0.9922,0.9992
0,Logistic Regression,0.9866,0.9830,0.9924,0.9876,0.9988
2,Random Forest,0.9764,0.9680,0.9890,0.9784,0.9977



=== Test Metrics (held-out generalization estimate) ===


,Model,accuracy,precision,recall,f1,auc_roc
1,Svm,0.9943,0.9957,0.9938,0.9947,0.9998
3,Neural Network,0.9941,0.9943,0.9947,0.9945,0.9998
0,Logistic Regression,0.9900,0.9891,0.9924,0.9907,0.9994
2,Random Forest,0.9820,0.9765,0.9904,0.9834,0.9990


In [13]:
# Generate ROC curves for all models
import os
OUTPUT_DIR = "./evaluation_outputs/v2_detailed"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def get_scores(model, X):
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    elif hasattr(model, "decision_function"):
        d = model.decision_function(X)
        return 1 / (1 + np.exp(-d))
    return model.predict(X).astype(float)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot([0, 1], [0, 1], "k--", label="Random baseline (AUC=0.50)")

all_predictions = {}
for name, path in saved_model_paths.items():
    model = load_model(path)
    y_pred = model.predict(X_test_full)
    y_score = get_scores(model, X_test_full)
    all_predictions[name] = y_pred
    fpr, tpr, _ = roc_curve(y_test, y_score)
    auc_val = roc_auc_score(y_test, y_score)
    ax.plot(fpr, tpr, lw=2, label=f"{name.replace('_', ' ').title()} (AUC={auc_val:.4f})")

ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve Comparison — All Candidate Models (Test Split)")
ax.legend(loc="lower right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/roc_curve_comparison.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved to {OUTPUT_DIR}/roc_curve_comparison.png")

Saved to ./evaluation_outputs/v2_detailed/roc_curve_comparison.png


In [14]:
# Generate confusion matrices for all models
fig, axes = plt.subplots(1, len(all_predictions), figsize=(5 * len(all_predictions), 5))
if len(all_predictions) == 1:
    axes = [axes]

for ax, (name, y_pred) in zip(axes, all_predictions.items()):
    cm = confusion_matrix(y_test, y_pred)
    im = ax.imshow(cm, interpolation="nearest", cmap="Blues")
    ax.set_title(name.replace('_', ' ').title(), fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_xticks([0, 1])
    ax.set_yticks([0, 1])
    ax.set_xticklabels(["FAKE", "REAL"])
    ax.set_yticklabels(["FAKE", "REAL"])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax.text(j, i, format(cm[i, j], "d"), ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")

fig.suptitle("Confusion Matrices — All Candidates (Test Split)", fontweight="bold", y=1.02)
plt.tight_layout()

# Also save individual confusion matrix for the best model (SVM)
for name, y_pred in all_predictions.items():
    cm = confusion_matrix(y_test, y_pred)
    fig_single, ax_single = plt.subplots(figsize=(6, 5))
    im = ax_single.imshow(cm, interpolation="nearest", cmap="Blues")
    plt.colorbar(im, ax=ax_single)
    ax_single.set_title(f"Confusion Matrix — {name.replace('_', ' ').title()}", fontweight="bold")
    ax_single.set_xlabel("Predicted")
    ax_single.set_ylabel("True")
    ax_single.set_xticks([0, 1])
    ax_single.set_yticks([0, 1])
    ax_single.set_xticklabels(["FAKE (0)", "REAL (1)"])
    ax_single.set_yticklabels(["FAKE (0)", "REAL (1)"])
    thresh = cm.max() / 2.0
    for i in range(2):
        for j in range(2):
            ax_single.text(j, i, format(cm[i, j], "d"), ha="center", va="center",
                           color="white" if cm[i, j] > thresh else "black")
    plt.tight_layout()
    fig_single.savefig(f"{OUTPUT_DIR}/confusion_matrix_{name}.png", dpi=300, bbox_inches="tight")
    plt.close(fig_single)

plt.show()
print(f"\nIndividual confusion matrices saved to {OUTPUT_DIR}/")


Individual confusion matrices saved to ./evaluation_outputs/v2_detailed/


In [15]:
# Classification reports for all models
for name, y_pred in all_predictions.items():
    print(f"\n{'='*50}")
    print(f"Classification Report: {name.replace('_', ' ').title()}")
    print(f"{'='*50}")
    print(classification_report(y_test, y_pred, target_names=["FAKE", "REAL"], digits=4))


Classification Report: Logistic Regression
              precision    recall  f1-score   support

        FAKE     0.9910    0.9872    0.9891      1790
        REAL     0.9891    0.9924    0.9907      2094

    accuracy                         0.9900      3884
   macro avg     0.9900    0.9898    0.9899      3884
weighted avg     0.9900    0.9900    0.9900      3884


Classification Report: Svm
              precision    recall  f1-score   support

        FAKE     0.9928    0.9950    0.9939      1790
        REAL     0.9957    0.9938    0.9947      2094

    accuracy                         0.9943      3884
   macro avg     0.9942    0.9944    0.9943      3884
weighted avg     0.9943    0.9943    0.9943      3884


Classification Report: Random Forest
              precision    recall  f1-score   support

        FAKE     0.9886    0.9721    0.9803      1790
        REAL     0.9765    0.9904    0.9834      2094

    accuracy                         0.9820      3884
   macro avg     0

---
## 7. Model Selection with Written Justification

### Decision Rule
Candidates are ranked by a **lexicographic key on the validation split only**:

```python
best = max(candidates, key=lambda name: (val_results[name]["f1"], val_results[name]["auc_roc"]))
```

### Justification
- **Primary criterion: F1-score** — fake-news classification requires balancing precision (avoiding false alarms that erode user trust) and recall (catching real misinformation before it spreads). F1 captures this balance.
- **Tie-breaker: AUC-ROC** — when F1 scores are within rounding distance, AUC-ROC favours the model whose probability ranking generalises better, which matters for the downstream probabilistic aggregation with the LLM verdict.
- **Validation split only** — the test split is reserved as a held-out estimate of generalisation for the final report. Selecting on test would leak the test set into the model-choice decision.

In [16]:
# Apply the selection logic
selected_model_name = train_output["selected_model_name"]
selected_val_metrics = candidate_validation_results[selected_model_name]
selected_test_metrics = candidate_test_results[selected_model_name]

# Sort candidates by (F1, AUC-ROC) descending
ranked = sorted(
    candidate_validation_results.items(),
    key=lambda x: (x[1]["f1"], x[1]["auc_roc"]),
    reverse=True,
)

print("=== Model Ranking (by Validation F1 -> AUC-ROC) ===")
for i, (name, metrics) in enumerate(ranked, 1):
    marker = " <-- SELECTED" if name == selected_model_name else ""
    print(f"  {i}. {name.replace('_', ' ').title():25s} F1={metrics['f1']:.4f}  AUC={metrics['auc_roc']:.4f}{marker}")

print(f"\n=== Selected Model: {selected_model_name.replace('_', ' ').title()} ===")
print(f"Validation metrics: {selected_val_metrics}")
print(f"Test metrics:       {selected_test_metrics}")

if len(ranked) > 1:
    runner_up_name, runner_up_metrics = ranked[1]
    print(f"\nRunner-up: {runner_up_name.replace('_', ' ').title()}")
    print(f"  Val F1:  {runner_up_metrics['f1']:.4f} vs winner {selected_val_metrics['f1']:.4f}")
    print(f"  Val AUC: {runner_up_metrics['auc_roc']:.4f} vs winner {selected_val_metrics['auc_roc']:.4f}")

print(f"\nModel path: {saved_model_paths[selected_model_name]}")

=== Model Ranking (by Validation F1 -> AUC-ROC) ===
  1. Neural Network            F1=0.9926  AUC=0.9993 <-- SELECTED
  2. Svm                       F1=0.9922  AUC=0.9992
  3. Logistic Regression       F1=0.9876  AUC=0.9988
  4. Random Forest             F1=0.9784  AUC=0.9977

=== Selected Model: Neural Network ===
Validation metrics: {'accuracy': 0.992, 'precision': 0.9907, 'recall': 0.9945, 'f1': 0.9926, 'auc_roc': np.float64(0.9993)}
Test metrics:       {'accuracy': 0.9941, 'precision': 0.9943, 'recall': 0.9947, 'f1': 0.9945, 'auc_roc': np.float64(0.9998)}

Runner-up: Svm
  Val F1:  0.9922 vs winner 0.9926
  Val AUC: 0.9992 vs winner 0.9993

Model path: ./models/v2/neural_network.joblib


---
## 8. Inference on a New Input (ML Only)

Before running the full LangGraph pipeline, we demonstrate standalone ML inference on a new article. This loads the selected model and applies the same TF-IDF + handcrafted feature pipeline used during training.

In [17]:
from src.utils.preprocessing import clean_text_for_traditional_ml
from src.utils.ingestion_tools import calculate_article_scores

# Load the selected model
selected_model = load_model(saved_model_paths[selected_model_name])

# New article to classify
new_article = (
    "WASHINGTON (Reuters) - The U.S. Federal Reserve kept interest rates unchanged on Wednesday "
    "and signaled it still planned three rate cuts for 2024, despite recent data showing inflation "
    "remains sticky. Fed Chair Jerome Powell said during a press conference that policymakers still "
    "believe inflation is on a path back to the central bank's 2% target, but acknowledged the "
    "journey may take longer than initially expected. The decision was widely anticipated by markets, "
    "which rallied following the announcement."
)

# 1. Clean text for ML
cleaned_text = clean_text_for_traditional_ml(new_article)

# 2. TF-IDF transform
X_text = tfidf.transform([cleaned_text])

# 3. Handcrafted features
features = calculate_article_scores(new_article)
numeric_values = [features.get(col, 0) for col in numeric_feature_cols]
# Convert bool to int
numeric_values = [int(v) if isinstance(v, bool) else v for v in numeric_values]
X_num = np.array([numeric_values], dtype=float)
X_num_scaled = scaler.transform(X_num)

# 4. Stack and predict
X_final = hstack([X_text, csr_matrix(X_num_scaled)])
ml_score = float(selected_model.predict_proba(X_final)[:, 1][0])  # P(REAL)
ml_label = "REAL" if ml_score >= 0.5 else "FAKE"
ml_confidence = ml_score if ml_label == "REAL" else 1 - ml_score

print(f"=== ML Inference Result ===")
print(f"Model:      {selected_model_name.replace('_', ' ').title()}")
print(f"Label:      {ml_label}")
print(f"P(REAL):    {ml_score:.4f}")
print(f"Confidence: {ml_confidence:.2%}")
print(f"\nHandcrafted features: {features}")

=== ML Inference Result ===
Model:      Neural Network
Label:      REAL
P(REAL):    1.0000
Confidence: 100.00%

Handcrafted features: {'sub_variance': 0.0004, 'mean_subjectivity': 0.2333, 'lexical_density': 0.8961, 'caps_ratio': 0.0433, 'has_dateline': True}


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


---
## 9. Downstream Skill Output — Full LangGraph Pipeline

Now we run the **complete agentic pipeline** on the same article. This exercises all 9 LangGraph nodes:

1. **Ingest** — extract text and stylistic features
2. **Preprocess** — load cached artifacts
3. **Train** — skip (cached)
4. **Evaluate** — skip (cached)
5. **Select Model** — load cached winner
6. **ML Classify** — TF-IDF + handcrafted features → P(REAL)
7. **LLM Classify** — GPT-4o-mini ReACT agent with sentiment, credibility, and cross-reference tools
8. **Reasoning Evaluate** — DeepEval GEval metric scores the LLM's reasoning quality
9. **Aggregate** — eval-score-weighted blend of ML and LLM signals → final verdict

In [18]:
from src.graph import run_agent
import time

# Run the full pipeline
article_text = (
    "WASHINGTON (Reuters) - The U.S. Federal Reserve kept interest rates unchanged on Wednesday "
    "and signaled it still planned three rate cuts for 2024, despite recent data showing inflation "
    "remains sticky. Fed Chair Jerome Powell said during a press conference that policymakers still "
    "believe inflation is on a path back to the central bank's 2% target, but acknowledged the "
    "journey may take longer than initially expected. The decision was widely anticipated by markets, "
    "which rallied following the announcement."
)

t0 = time.time()
result = run_agent(article_text, input_type="text")
elapsed = time.time() - t0

print(f"\n{'='*60}")
print(f"FULL PIPELINE RESULT (completed in {elapsed:.1f}s)")
print(f"{'='*60}")
print(f"Final Verdict:    {result['final_label']}")
print(f"Final Confidence: {result['final_score']:.2%}")
print(f"\n--- Phase 1: ML Classification ---")
print(f"ML Label:  {result['ml_label']}")
print(f"ML Score:  {result['ml_score']:.4f} (P(REAL))")
ml_conf = result['ml_score'] if result['ml_label'] == 'REAL' else 1 - result['ml_score']
print(f"ML Confidence: {ml_conf:.2%}")
print(f"\n--- Phase 2: LLM ReACT Agent ---")
print(f"LLM Label:  {result['llm_label']}")
print(f"LLM Score:  {result['llm_score']:.2%}")
print(f"\n--- Reasoning Evaluation (DeepEval) ---")
print(f"Eval Score: {result['eval_score']:.2f}/1.00")
print(f"\n--- Aggregation ---")
print(f"ML Weight:  {result.get('ml_weight', 'N/A')}")
print(f"LLM Weight: {result.get('llm_weight', 'N/A')}")
print(f"Agreement:  {result.get('eval_agreement', 'N/A')}")
print(f"\n--- Summary ---")
print(result['summary'])
print(f"\n--- LLM Reasoning ---")
print(result.get('llm_reasoning', 'No reasoning provided.'))


>>> [NODE] Starting Ingestion Node...
>>> [NODE] Finished Ingestion Node.

>>> [NODE] Starting Preprocess Data Node...
>>> [LOG] v2 preprocessing artifacts found at ./models/v2/preprocessing_artifacts.joblib. Skipping processing.
>>> [NODE] Finished Preprocess Data Node.

>>> [NODE] Starting Train Models Node...
>>> [LOG] v2 training artifacts found at ./models/v2/training_artifacts.joblib. Skipping training.
>>> [NODE] Finished Train Models Node.

>>> [NODE] Starting Evaluate Models Node...
>>> [LOG] Skipping evaluation — v2 training cache is present.
>>> [NODE] Finished Evaluate Models Node.

>>> [NODE] Starting Select Model Node...
>>> [LOG] Using cached v2 training artifacts at ./models/v2/training_artifacts.joblib.
>>> [NODE] Finished Select Model Node.

>>> [NODE] Starting ML Classifier Node...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


>>> [NODE] Finished ML Classifier Node.

>>> [NODE] Starting LLM Classifier Node...
[llm_classifier] Agent made 3 tool call(s):
  [1] sentiment_analysis_tool args={'text': "WASHINGTON (Reuters) - The U.S. Federal Reserve kept interest rates unchanged on Wednesday and signaled it still planned three rate cuts for 2024, despite recent data showing inflation remains sticky. Fed Chair Jerome Powell said during a press conference that policymakers still believe inflation is on a path back to the central bank's 2% target, but acknowledged the journey may take longer than initially expected. The decision was widely anticipated by markets, which rallied following the announcement."}
      -> Emotional Tone: neutral | Tone Score: 0.1323 (0=neutral, 1=sensationalist) | Polarity: -0.0333 | Subjectivity: 0.2333 | Exclamation Ratio: 0.0 | ALL-CAPS Word Ratio: 0.026 | Sentence Count: 3
  [2] source_credibility_tool args={'url': 'https://www.reuters.com'}
      -> Domain: reuters.com | Credibility Sc

Output()

>>> [NODE] Finished Evaluator Node.

>>> [NODE] Starting Aggregator Node...
>>> [NODE] Finished Aggregator Node.

FULL PIPELINE RESULT (completed in 33.1s)
Final Verdict:    REAL
Final Confidence: 96.00%

--- Phase 1: ML Classification ---
ML Label:  REAL
ML Score:  1.0000 (P(REAL))
ML Confidence: 100.00%

--- Phase 2: LLM ReACT Agent ---
LLM Label:  REAL
LLM Score:  92.00%

--- Reasoning Evaluation (DeepEval) ---
Eval Score: 0.83/1.00

--- Aggregation ---
ML Weight:  0.5
LLM Weight: 0.5
Agreement:  True

--- Summary ---
The article is classified as REAL with 0.96 confidence. LLM reasoning quality was high — equal weight applied.

--- LLM Reasoning ---
The article reports on the U.S. Federal Reserve's decision to keep interest rates unchanged, which is corroborated by 5 independent sources including ABC News and Fortune. The tone analysis indicates a neutral tone with a tone score of 0.1323, suggesting no sensationalism. The source, Reuters, has a high credibility tier with a score of 

In [19]:
# Test with a fake article
fake_article = (
    "EXPOSED: Government Officials Caught RED-HANDED Hiding Secret Alien Technology!!! "
    "Whistle-blowers from deep inside the Pentagon have FINALLY come forward to reveal that our "
    "government has been LYING to us for DECADES about contact with extraterrestrial beings!!! "
    "The mainstream media is REFUSING to cover this story because they are CONTROLLED by the very "
    "same elites who want to keep this information from YOU!!! Share this before they DELETE it!!!"
)

t0 = time.time()
result_fake = run_agent(fake_article, input_type="text")
elapsed = time.time() - t0

print(f"\n{'='*60}")
print(f"FAKE ARTICLE PIPELINE RESULT (completed in {elapsed:.1f}s)")
print(f"{'='*60}")
print(f"Final Verdict:    {result_fake['final_label']}")
print(f"Final Confidence: {result_fake['final_score']:.2%}")
print(f"ML:  {result_fake['ml_label']} (P(REAL)={result_fake['ml_score']:.4f})")
print(f"LLM: {result_fake['llm_label']} ({result_fake['llm_score']:.2%})")
print(f"Eval Score: {result_fake['eval_score']:.2f}")
print(f"Agreement:  {result_fake.get('eval_agreement', 'N/A')}")
print(f"\nSummary: {result_fake['summary']}")
print(f"\nReasoning: {result_fake.get('llm_reasoning', 'N/A')}")


>>> [NODE] Starting Ingestion Node...
>>> [NODE] Finished Ingestion Node.

>>> [NODE] Starting Preprocess Data Node...
>>> [LOG] v2 preprocessing artifacts found at ./models/v2/preprocessing_artifacts.joblib. Skipping processing.
>>> [NODE] Finished Preprocess Data Node.

>>> [NODE] Starting Train Models Node...
>>> [LOG] v2 training artifacts found at ./models/v2/training_artifacts.joblib. Skipping training.
>>> [NODE] Finished Train Models Node.

>>> [NODE] Starting Evaluate Models Node...
>>> [LOG] Skipping evaluation — v2 training cache is present.
>>> [NODE] Finished Evaluate Models Node.

>>> [NODE] Starting Select Model Node...
>>> [LOG] Using cached v2 training artifacts at ./models/v2/training_artifacts.joblib.
>>> [NODE] Finished Select Model Node.

>>> [NODE] Starting ML Classifier Node...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


>>> [NODE] Finished ML Classifier Node.

>>> [NODE] Starting LLM Classifier Node...
[llm_classifier] Agent made 3 tool call(s):
  [1] sentiment_analysis_tool args={'text': 'EXPOSED: Government Officials Caught RED-HANDED Hiding Secret Alien Technology!!! Whistle-blowers from deep inside the Pentagon have FINALLY come forward to reveal that our government has been LYING to us for DECADES about contact with extraterrestrial beings!!! The mainstream media is REFUSING to cover this story because they are CONTROLLED by the very same elites who want to keep this information from YOU!!! Share this before they DELETE it!!!'}
      -> Emotional Tone: highly_emotional | Tone Score: 0.7395 (0=neutral, 1=sensationalist) | Polarity: -0.1777 | Subjectivity: 0.6025 | Exclamation Ratio: 1.0 | ALL-CAPS Word Ratio: 0.1324 | Sentence Count: 5
  [2] source_credibility_tool args={'url': ''}
      -> Domain:  | Credibility Score: 0.5 | Credibility Tier: unknown | Known Source: False | Warning: No URL provid

Output()

>>> [NODE] Finished Evaluator Node.

>>> [NODE] Starting Aggregator Node...
>>> [NODE] Finished Aggregator Node.

FAKE ARTICLE PIPELINE RESULT (completed in 19.6s)
Final Verdict:    FAKE
Final Confidence: 97.50%
ML:  FAKE (P(REAL)=0.0000)
LLM: FAKE (95.00%)
Eval Score: 0.91
Agreement:  True

Summary: The article is classified as FAKE with 0.97 confidence. LLM reasoning quality was high — equal weight applied.

Reasoning: The article exhibits a highly emotional tone with a tone score of 0.7395, indicating sensationalism. The use of all-caps and exclamation marks further amplifies this effect, with an exclamation ratio of 1.0 and an ALL-CAPS word ratio of 0.1324. Additionally, the source credibility is unknown, as no URL was provided, and there were zero corroborating articles found in the cross-reference search, suggesting that the claims may be fabricated or too niche. Given the lack of credible sources and the sensationalist style, the evidence strongly supports a verdict of FAKE.


---
## 10. LangGraph Pipeline Definition

The pipeline is wired as a **linear StateGraph** in `src/graph.py`. Each node reads from and writes to a shared `AgentState` TypedDict. Below is the full pipeline definition alongside all SKILL.md content that governs each node's behaviour.

In [20]:
# Display the LangGraph pipeline definition
print("=" * 60)
print("LangGraph Pipeline Definition (src/graph.py)")
print("=" * 60)
with open("src/graph.py", "r") as f:
    print(f.read())

LangGraph Pipeline Definition (src/graph.py)
"""
graph.py — LangGraph Pipeline Wiring

Imports all nodes from src/nodes/ and wires them into a StateGraph.

This is the only file that knows about the full pipeline topology.
"""

from langgraph.graph import StateGraph, END
from src.state import AgentState
from src.nodes import (
    ingestion_node,
    preprocess_data_node,
    train_models_node,
    evaluate_models_node,
    select_model_node,
    ml_classifier_node,
    llm_classifier_node,
    evaluator_node,
    aggregator_node,
)

# ── Build the graph ──────────────────────────────────────────
builder = StateGraph(AgentState)

builder.add_node("ingest", ingestion_node)
builder.add_node("preprocess_data", preprocess_data_node)
builder.add_node("train_models", train_models_node)
builder.add_node("evaluate_models", evaluate_models_node)
builder.add_node("select_model", select_model_node)
builder.add_node("ml_classify", ml_classifier_node)
builder.add_node("llm_classify", llm_classifier

In [21]:
# Display all SKILL.md files that govern node behaviour
import glob

skill_files = sorted(glob.glob("skills/*.md"))
print(f"Found {len(skill_files)} skill files:\n")
for skill_path in skill_files:
    print("=" * 60)
    print(f"SKILL: {skill_path}")
    print("=" * 60)
    with open(skill_path, "r") as f:
        print(f.read())
    print()

Found 9 skill files:

SKILL: skills/aggregation.md
---
name: aggregation
description: Combine the ML classifier's probability and the LLM classifier's verdict into a single final label, confidence, and user-facing summary for the Gradio UI.
mode: organisational
---

# Aggregation Skill

You are the Aggregator Agent. Your job is to merge the ML and LLM signals into one final verdict and produce the business-facing summary and explanation blocks that the Gradio interface renders to the end user.

## When to use
- Once, at the very end of the inference pipeline, after ML Classification, LLM Classification, and Evaluation have all completed.
- Always — this stage is required for producing the UI response.

## How to execute
1. **Thought**: Check whether the ML and LLM labels agree, and how wide the confidence gap is.
2. **Action**:
   - Convert both signals to `P(REAL)` on [0, 1]. `ml_score` is already `P(REAL)`; map `llm_score` via `llm_score if llm_label == "REAL" else 1 - llm_score`.
  

In [22]:
# Display the AgentState definition
print("=" * 60)
print("AgentState Definition (src/state.py)")
print("=" * 60)
with open("src/state.py", "r") as f:
    print(f.read())

AgentState Definition (src/state.py)
"""
state.py — Shared Agent State

Single source of truth for the data flowing through the LangGraph pipeline.
Every node reads from and writes to this state.
"""

from typing import TypedDict, Optional, Literal, List, Dict, Any


class AgentState(TypedDict, total=False):
    """State shared across all LangGraph nodes."""

    # ── Raw input (set once at pipeline entry) ──────────────────
    input_type: Literal["text", "url", "file"]   # how the user provided the article
    raw_input: str                                # original user input (text / URL / file path)

    # ── Dataset / training configuration ───────────────────────
    fake_csv_path: str
    true_csv_path: str
    train_size: float
    val_size: float
    test_size: float
    random_state: int

    include_transformer: bool
    transformer_model_name: str
    transformer_epochs: int
    transformer_batch_size: int
    transformer_learning_rate: float

    # ── Preprocessing outputs

### Pipeline Architecture

```
START (User Input)
  │
  ▼
┌─────────────────┐
│  Ingest Node     │  Extract text + stylistic features (caps_ratio, lexical_density, etc.)
│  SKILL: ingestion│
└────────┬────────┘
         ▼
┌─────────────────┐
│  Preprocess Node │  Load cached V2 artifacts (canonical dedup, stratified splits)
│  SKILL: preproc  │
└────────┬────────┘
         ▼
┌─────────────────┐
│  Train Node      │  Fit 4 ML candidates (LR, SVM, RF, MLP) — skipped if cached
│  SKILL: train    │
└────────┬────────┘
         ▼
┌─────────────────┐
│  Evaluate Node   │  Compute metrics + ROC/CM visualisations — skipped if cached
│  SKILL: evaluate │
└────────┬────────┘
         ▼
┌─────────────────┐
│  Select Node     │  Pick winner by (F1, AUC-ROC) on validation split
│  SKILL: select   │
└────────┬────────┘
         ▼
┌─────────────────┐
│  ML Classify     │  Phase 1: TF-IDF + features → P(REAL)
│  SKILL: ml_class │
└────────┬────────┘
         ▼
┌─────────────────┐
│  LLM Classify    │  Phase 2: GPT-4o-mini ReACT agent with tools
│  SKILL: llm_class│  (sentiment, credibility, cross-reference)
└────────┬────────┘
         ▼
┌─────────────────┐
│  Reasoning Eval  │  DeepEval GEval scores LLM reasoning quality
│  SKILL: eval     │
└────────┬────────┘
         ▼
┌─────────────────┐
│  Aggregator      │  Eval-score weighted blend → final REAL/FAKE verdict
│  SKILL: agg      │
└────────┬────────┘
         ▼
       END (Verdict + Explanation)
```

---
## 11. Gradio Interface Launch (`share=True`)

Launches the full interactive UI with a public Gradio share link. The UI provides:
- Text paste or URL input
- Real-time pipeline status updates
- Verdict display with confidence scores
- Detailed analysis (key signals, tool trace, pipeline breakdown)
- Full AI explanation with corroborating sources
- Model Performance Dashboard (ROC curves, confusion matrices, metric table)

In [25]:
# Patch main.py for Colab compatibility
import re

with open("main.py", "r") as f:
    content = f.read()

# 1. share=True for Colab public link
content = content.replace("share=False", "share=True")

# 2. Remove inbrowser=True (doesn't work on Colab)
content = content.replace("inbrowser=True,\n", "")
content = content.replace("inbrowser=True,", "")

# 3. Move css= and theme= from launch() into gr.Blocks() constructor
#    Newer Gradio versions don't accept css/theme in launch()
content = content.replace(
    'title="Fake News Detection Agent",\n) as demo:',
    'title="Fake News Detection Agent",\n    css=CUSTOM_CSS,\n    theme=theme,\n) as demo:'
)
# Remove css= and theme= from launch()
content = re.sub(r'\s*css=CUSTOM_CSS,\n', '\n', content)
content = re.sub(r'\s*theme=theme,\n', '\n', content)

with open("main.py", "w") as f:
    f.write(content)

print("Patched main.py for Colab (share=True, css/theme moved to gr.Blocks).")

Patched main.py for Colab (share=True, css/theme moved to gr.Blocks).


In [26]:
# Launch the Gradio app — a public *.gradio.live URL will appear below
!python main.py

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://2d9c258698d0f22b13.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)

>>> [NODE] Starting Ingestion Node...
>>> [NODE] Finished Ingestion Node.

>>> [NODE] Starting Preprocess Data Node...
>>> [LOG] v2 preprocessing artifacts found at ./models/v2/preprocessing_artifacts.joblib. Skipping processing.
>>> [NODE] Finished Preprocess Data Node.

>>> [NODE] Starting Train Models Node...
>>> [LOG] v2 training artifacts found at ./models/v2/training_artifacts.joblib. Skipping training.
>>> [NODE] Finished Train Models Node.

>>> [NODE] Starting Evaluate Models Node...
>>> [LOG] Skipping evaluation — v2 training cache is present.
>>> [NODE] Finished Evaluate Models Node.

>>> [NODE] Starting Select Model Node...
>>> [LOG] Using cached v2 training a